# 🐜 Ant Colony Simulation

**Authors:** [Your names]

---

## 📋 Scenario Description

We simulate an ant colony facing a roaming predator. The colony consists of three roles: **foragers** that collect food pellets and bring energy back to the colony; **fighters** that patrol the colony perimeter and charge at the predator when it gets close; and a **queen** who sits stationary at the colony centre.

A single **predator** roams the map and hunts ants. All agents operate under an internal **energy** level that drains over time and gates behaviour intensity — a hungry predator hunts more aggressively, a stressed forager stops foraging and flees instead. Ants consumed by the predator are temporarily removed and respawn at the queen after a stun delay.

We expect to observe emergent colony-level patterns from these simple local rules: foragers oscillating between foraging and fleeing as the predator approaches; fighters clustering around the queen when an alarm is triggered; and predator energy driving rhythmic hunt-and-idle cycles.

---

## 🗺️ Entity Plan

| Entity type | Subtype label | Count | Color | Diameter | Role |
|------------|--------------|-------|-------|----------|---------|
| Agent | `"forager"` | 6 | goldenrod | 3 | Collect food, flee predator, return to queen |
| Agent | `"fighter"` | 4 | orangered | 4 | Patrol colony, attack predator on alarm |
| Agent | `"queen"` | 1 | mediumpurple | 7 | Stationary colony anchor, home reference |
| Agent | `"predator"` | 1 | crimson | 8 | Roaming threat, hunts foragers and fighters |
| Object | `"food"` | 24 | limegreen | 3 | Energy source, consumed by foragers, respawns |
| Object | `"obstacle"` | 8 | saddlebrown | 10 | Immovable blocks, navigation constraints |


---
## ⚙️ Section 1 — Setup

In [ ]:
from vivarium.controllers import VivariumController
controller = VivariumController.start_session(scene_name="miniproject")

In [ ]:
# ── Define your subtype labels here (max 8) ──────────────────────────────────
# Replace the example labels below with labels that match YOUR scenario
controller.set_subtype_labels([
    "type_a",     # e.g. prey / bee / robot
    "type_b",     # e.g. predator / hive / base
    "resource",   # e.g. food / flower / pollen
    "obstacle",   # e.g. wall / rock / border
    # Add up to 4 more if needed
])

# Verify
print("Available subtypes:", controller.subtypes)

In [ ]:
# ── (Optional) Speed calibration ─────────────────────────────────────────────
# Increase if simulation feels too slow. Keep between 1 and 6.
controller.simulator.env.num_scan_steps = 2

---
## 🧩 Section 2 — Entity Initialization

Assign subtypes, colors, sizes, and speeds to all 12 agents and 32 objects according to the entity plan above.

**Step 1 goal:** get all entities visible on the map with the right appearance before adding any behaviour. Agents `[0:6]` → foragers · `[6:10]` → fighters · `[10]` → queen (place at map centre, make immovable) · `[11]` → predator. Objects `[0:24]` → food · `[24:32]` → obstacles (make immovable).  
*(Agents = squares, Objects = circles on the map)*

In [ ]:
# ── Agent initialization ──────────────────────────────────────────────────────
# 12 agents total: controller.agents[0:12]

# -- Group A agents (e.g. first 8) --
for agent in controller.agents[0:8]:
    agent.subtype = "type_a"
    agent.diameter = 3
    agent.color = "blue"
    agent.max_speed = 2
    # agent.proxs_dist_max = 20     # sensing range
    # agent.proxs_cos_min  = 0.0    # field of view (-1=360°, 0=180°, 1=0°)

# -- Group B agents (e.g. last 4) --
for agent in controller.agents[8:12]:
    agent.subtype = "type_b"
    agent.diameter = 6
    agent.color = "red"
    agent.max_speed = 1

print("Agents initialized.")

In [ ]:
# ── Object initialization ─────────────────────────────────────────────────────
# 32 objects total: controller.objects[0:32]

# -- Resources (e.g. first 24) --
for obj in controller.objects[0:24]:
    obj.subtype = "resource"
    obj.diameter = 3
    obj.color = "green"

# -- Obstacles (e.g. next 8) --
for obj in controller.objects[24:32]:
    obj.subtype = "obstacle"
    obj.diameter = 8
    obj.color = "dimgray"
    obj.mass = 1000       # make immovable
    obj.friction = 1000

print("Objects initialized.")

In [ ]:
# ── Quick verification ────────────────────────────────────────────────────────
print("=== Agents ===")
for agent in controller.agents:
    print(f"  subtype={agent.subtype}, diameter={agent.diameter}, color={agent.color}, max_speed={agent.max_speed}")

print("\n=== Objects (sample: first and last 4) ===")
for obj in list(controller.objects[0:4]) + list(controller.objects[28:32]):
    print(f"  subtype={obj.subtype}, diameter={obj.diameter}, color={obj.color}")

---
## 🤖 Section 3 — Behaviors

**Step 1 scope — foragers, food, obstacles (no predator yet):** implement `avoid_obstacles` and `forage` for foragers, and `wander` as a fallback when no food is in range. Verify foragers navigate around obstacles and collect food.

**Step 2 scope — add the predator:** implement `hunt_ants` for the predator (always hunting at this stage, no energy gating yet) and `flee_predator` for foragers. Add `patrol_colony` and `attack_predator` for fighters. Verify the predator chases ants and fighters respond.

Each behaviour function takes an `agent` and returns `(left_motor, right_motor)` in `[0, 1]`. Weights set here are the starting values; Section 4 routines will adjust them dynamically.

In [ ]:
# ── Behavior definitions ──────────────────────────────────────────────────────
# Each function takes an `agent` and returns (left_motor, right_motor) in [0, 1]

# Template: obstacle avoidance (shyness — crossed inhibitory)
def avoid_obstacles(agent):
    left, right = agent.proximeters(sensed_entities=["obstacle"])
    return 1 - right, 1 - left

# Template: attraction toward resources (aggression — crossed excitatory)
def forage_resources(agent):
    left, right = agent.proximeters(sensed_entities=["resource"])
    return right, left

# Template: flee from type_b (fear — direct excitatory)
def flee_from_b(agent):
    left, right = agent.proximeters(sensed_entities=["type_b"])
    return left, right

# Template: chase type_a (aggression — crossed excitatory)
def chase_a(agent):
    left, right = agent.proximeters(sensed_entities=["type_a"])
    return right, left

# ── Add your own behaviors below ──────────────────────────────────────────────
# def my_custom_behavior(agent):
#     left, right = agent.proximeters(sensed_entities=["..."])
#     ...
#     return left_motor, right_motor

print("Behaviors defined.")

In [ ]:
# ── Attach behaviors to agents ────────────────────────────────────────────────
# Always detach first to avoid duplicates when re-running this cell
for agent in controller.agents:
    agent.detach_all_behaviors(stop_motors=True)
    agent.detach_all_routines()

# -- type_a agents --
for agent in controller.agents:
    if agent.subtype == "type_a":
        agent.attach_behavior(avoid_obstacles, weight=1.0)
        agent.attach_behavior(forage_resources, weight=1.0)
        agent.attach_behavior(flee_from_b, weight=1.5)  # stronger flight impulse

# -- type_b agents --
for agent in controller.agents:
    if agent.subtype == "type_b":
        agent.attach_behavior(avoid_obstacles, weight=1.0)
        agent.attach_behavior(chase_a, weight=1.0)

print("Behaviors attached.")

---
## 🔄 Section 4 — Internal States & Routines

**Step 3 scope — energy-weighted behaviours:** introduce `energy` as the primary drive modulator for all agents, and `stress` for foragers only.

- **Energy** drains each tick at a fixed rate per agent type. Foragers and fighters recover energy by consuming food; the predator recovers by consuming ants. At low energy, foraging/hunting weights increase; at high energy, idle/wander weights dominate.
- **Stress** (foragers only) accumulates when the predator is nearby and decays passively. Above the stress threshold, the `forage` weight drops and `flee_predator` weight spikes.
- Use `energy_diameter` as a visual health indicator — agents shrink as they lose energy.

Remember: initialise all `agent.internal` fields *before* attaching routines.

In [ ]:
# ── Initialize internal state variables BEFORE attaching routines ─────────────
for agent in controller.agents:
    if agent.subtype == "type_a":
        agent.internal.energy = 0.5  # starting energy level [0, 1]
        # agent.internal.age = 0      # add more fields as needed

print("Internal states initialized.")

In [ ]:
# ── Routine definitions ───────────────────────────────────────────────────────

# Energy routine: decays over time, recovers when consuming a resource
def energy_routine(agent):
    agent.internal.energy -= 0.0002                          # slow decay
    consumed = agent.has_consumed()                          # how many consumed this step
    agent.internal.energy += 0.1 * consumed                 # recover on consumption
    agent.internal.energy = max(0.0, min(1.0, agent.internal.energy))  # clamp [0,1]

# Visual feedback: map energy → diameter
def energy_diameter(agent):
    agent.diameter = 1 + agent.internal.energy * 4          # diameter 1–5

# Dynamic behavior weight: hungry agents forage harder
def dynamic_foraging_weight(agent):
    w = 2.0 - agent.internal.energy * 1.5                   # weight 0.5–2.0
    agent.change_behavior_weight(forage_resources, new_weight=w)

# ── Add your own routines below ───────────────────────────────────────────────
# def my_routine(agent):
#     ...

print("Routines defined.")

In [ ]:
# ── Attach routines to agents ─────────────────────────────────────────────────
for agent in controller.agents:
    if agent.subtype == "type_a":
        agent.attach_routine(energy_routine)
        agent.attach_routine(energy_diameter)
        agent.attach_routine(dynamic_foraging_weight)

print("Routines attached.")

---
## 🌱 Section 5 — Environmental Dynamics

Configure the consumption and spawning mechanisms that keep the simulation alive over time.

- **Slot 1 consumption:** foragers consume food pellets on contact — this is what triggers energy recovery in the forager energy routine.
- **Slot 2 consumption:** the predator consumes foragers — this removes the ant from the map (exists = False), which the stun/respawn controller routine in Section 6 will detect and handle.
- **Slot 3 consumption:** the predator consumes fighters — same stun logic applies.
- **Slot 1 spawn:** food pellets respawn periodically so the colony always has something to forage for. Tune the period to control how abundant food is.

In [ ]:
# ── Consumption slots ─────────────────────────────────────────────────────────

# Slot 1: type_a agents consume resources
controller.consumption.slot_1.source_subtype = "type_a"
controller.consumption.slot_1.target_subtype = "resource"
controller.consumption.slot_1.range = 1
controller.consumption.slot_1.start = True

# Slot 2: type_b agents consume type_a agents
controller.consumption.slot_2.source_subtype = "type_b"
controller.consumption.slot_2.target_subtype = "type_a"
controller.consumption.slot_2.range = 1
controller.consumption.slot_2.start = True

# Slots 3 and 4 available for additional consumption rules
# controller.consumption.slot_3.source_subtype = "..."
# controller.consumption.slot_3.target_subtype = "..."
# controller.consumption.slot_3.range = 1
# controller.consumption.slot_3.start = True

print("Consumption configured.")

In [ ]:
# ── Spawning slots ────────────────────────────────────────────────────────────

# Slot 1: resources respawn every 100 steps
controller.spawn.slot_1.subtype = "resource"
controller.spawn.slot_1.period = 100
# controller.spawn.slot_1.position_range = [10, 90, 10, 90]  # optional area constraint
controller.spawn.slot_1.start = True

# Slot 2: type_a agents respawn every 200 steps
controller.spawn.slot_2.subtype = "type_a"
controller.spawn.slot_2.period = 200
controller.spawn.slot_2.start = True

# Slots 3 and 4 available
# controller.spawn.slot_3.subtype = "..."
# controller.spawn.slot_3.period = ...
# controller.spawn.slot_3.start = True

print("Spawning configured.")

---
## 🔁 Section 6 — Controller Routines (Simulation-Wide Logic)

**Step 4 scope — colony complexity:** layer in colony-level emergent behaviour using controller routines that coordinate across multiple agents.

- **Stun & respawn:** when the predator consumes an ant, Vivarium sets `agent.exists = False`. This routine detects those ants, counts down a stun timer, then teleports them back to the queen with partial energy restored — simulating a stunned ant recovering at the colony.
- **Territorial alarm:** approximates the alarm pheromone from the project plan. If the predator enters a radius around the queen, all fighters immediately receive maximum attack weight — overriding the weight routine — regardless of how close they personally are to the predator.
- **Population logging:** records how many foragers, fighters, and food pellets exist each tick, plus predator energy, so we can plot colony dynamics afterwards.

In [ ]:
# ── Controller routine template: reproduction ─────────────────────────────────
def reproduction(controller):
    """When a type_a agent's energy exceeds 0.9, it spawns an offspring."""
    for agent in controller.agents:
        if agent.subtype == "type_a" and agent.exists and agent.internal.energy >= 0.9:
            # Find a non-existing type_a slot to use as offspring
            free_slots = [a for a in controller.agents if a.subtype == "type_a" and not a.exists]
            if not free_slots:
                return  # No room for offspring
            offspring = free_slots[0]
            offspring.exists = True
            offspring.x_position = agent.x_position + 1
            offspring.y_position = agent.y_position + 1
            offspring.internal.energy = 0.5
            agent.internal.energy -= 0.5   # parent pays the cost


# ── Controller routine template: population logging ───────────────────────────
def log_populations(controller):
    """Log population counts of each subtype every step."""
    n_a = len([a for a in controller.agents if a.subtype == "type_a" and a.exists])
    n_b = len([a for a in controller.agents if a.subtype == "type_b" and a.exists])
    n_r = len([o for o in controller.objects if o.subtype == "resource" and o.exists])
    controller.logger.add("type_a_count", n_a)
    controller.logger.add("type_b_count", n_b)
    controller.logger.add("resource_count", n_r)

print("Controller routines defined.")

In [ ]:
# ── Attach controller routines ────────────────────────────────────────────────
controller.attach_routine(reproduction)                    # every step
controller.attach_routine(log_populations, interval=10)   # every 10 steps

print("Controller routines attached.")
controller.print_routines()

---
## 📊 Section 7 — Data Logging & Plots

Visualize what is happening in the simulation using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# ── Plot population dynamics ──────────────────────────────────────────────────
# (Run this cell after the simulation has been running for a while)

plt.figure(figsize=(10, 4))
plt.plot(controller.logger.get("type_a_count"),   label="type_a agents", color="steelblue")
plt.plot(controller.logger.get("type_b_count"),   label="type_b agents", color="firebrick")
plt.plot(controller.logger.get("resource_count"), label="resources",     color="seagreen")
plt.xlabel("Time (×10 steps)")
plt.ylabel("Population")
plt.title("Population dynamics over time")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── (Optional) Log and plot individual agent energy ───────────────────────────

def log_energy(agent):
    agent.logger.add("energy", agent.internal.energy)

# Attach to a specific agent to track it
tracked_agent = controller.agents[0]
tracked_agent.attach_routine(log_energy, interval=10)

print(f"Now logging energy of agent 0 (subtype: {tracked_agent.subtype})")

In [ ]:
# Run this cell after some time to see the energy trace
energy_log = tracked_agent.logger.get("energy")
if len(energy_log) > 0:
    plt.figure(figsize=(10, 3))
    plt.plot(energy_log, color="darkorange")
    plt.xlabel("Time (×10 steps)")
    plt.ylabel("Energy level")
    plt.title("Energy level of tracked agent over time")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data yet — wait a moment and re-run.")

In [ ]:
# ── (Optional) Plot agent trajectory ─────────────────────────────────────────

def log_position(agent):
    agent.logger.add("x", agent.x_position)
    agent.logger.add("y", agent.y_position)

tracked_agent.attach_routine(log_position, interval=5)
print("Logging position of agent 0...")

In [ ]:
xs = tracked_agent.logger.get("x")
ys = tracked_agent.logger.get("y")
if len(xs) > 1:
    plt.figure(figsize=(5, 5))
    plt.plot(xs, ys, '.', markersize=2, alpha=0.6)
    plt.xlim(0, 100)
    plt.ylim(0, 100)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title("Trajectory of tracked agent")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough position data yet.")

---
## 🧹 Section 8 — Utilities

Handy one-off cells to inspect or reset the simulation state.

In [ ]:
# ── Inspect current state of all agents ───────────────────────────────────────
for agent in controller.agents:
    exists_str = "✓" if agent.exists else "✗"
    energy_str = f", energy={agent.internal.energy:.2f}" if hasattr(agent.internal, 'energy') else ""
    print(f"[{exists_str}] {agent.subtype:10s} | d={agent.diameter} | speed={agent.max_speed}{energy_str}")

In [ ]:
# ── Stop everything (use when experimenting) ──────────────────────────────────
for agent in controller.agents:
    agent.detach_all_behaviors(stop_motors=True)
    agent.detach_all_routines()

controller.consumption.slot_1.start = False
controller.consumption.slot_2.start = False
controller.spawn.slot_1.start = False
controller.spawn.slot_2.start = False

# Detach controller routines too
controller.detach_all_routines()

print("All behaviors, routines and mechanisms stopped.")

In [ ]:
# ── Clear all logged data ──────────────────────────────────────────────────────
for topic in ["type_a_count", "type_b_count", "resource_count"]:
    controller.logger.clear(topic)

for agent in controller.agents:
    for topic in ["energy", "x", "y"]:
        agent.logger.clear(topic)

print("Logs cleared.")

---
## 📝 Section 9 — Conclusion

> _Write your conclusions here after running the simulation._

**What did you observe?**  
> ...

**Did the simulation match your initial expectations?**  
> ...

**What would you have liked to add or improve?**  
> ...

**Limitations or issues encountered:**  
> ...

---

## 📚 Quick Reference

| Task | Code |
|------|------|
| Attach behavior | `agent.attach_behavior(fn, weight=1.0)` |
| Detach all behaviors | `agent.detach_all_behaviors(stop_motors=True)` |
| Attach routine | `agent.attach_routine(fn, interval=1)` |
| Detach all routines | `agent.detach_all_routines()` |
| Sense entities | `agent.proximeters(sensed_entities=["subtype"])` |
| Change behavior weight | `agent.change_behavior_weight(fn, new_weight=2.0)` |
| Check consumed | `agent.has_consumed()` |
| Log data | `agent.logger.add("topic", value)` |
| Retrieve data | `agent.logger.get("topic")` |
| Clear log | `agent.logger.clear("topic")` |
| Toggle existence | `agent.exists = True / False` |
| Set position | `agent.x_position = 50; agent.y_position = 50` |
| Consumption slot | `controller.consumption.slot_1.start = True/False` |
| Spawn slot | `controller.spawn.slot_1.start = True/False` |
| Controller routine | `controller.attach_routine(fn, interval=10)` |
